# Sentiment analysis of tweets: one-hot, BoW, TF-IDF, GloVe, BERT

*Corrected version (see Chapter 18, section 'Corrections to the slides and the notebook'). The original is kept as `Sentiment_Analysis_of_Tweets_original.ipynb`.*

# Loading libraries

In [ ]:
import re
import random
import numpy as np
import nltk
from nltk.corpus import twitter_samples
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED)
for pkg in ("twitter_samples", "punkt_tab"):
    nltk.download(pkg, quiet=True)

# Loading the dataset

In [ ]:
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')
print('Number of positive tweets:', len(all_positive_tweets))
print('Number of negative tweets:', len(all_negative_tweets))
print(all_positive_tweets[0])
print(all_negative_tweets[0])

## Warning: the labels are the emoticons

In this corpus a tweet is *positive* if it contains `:)` and *negative* if it contains `:(` (the idea of distant supervision). The label is therefore written **inside the text**. Any model that can see the emoticon will reach about 99.6 % without understanding any sentiment. The original notebook reported 0.997 for this reason. We remove the emoticons and then compare.

In [ ]:
tweets = all_positive_tweets + all_negative_tweets
y = np.array([1] * len(all_positive_tweets) + [0] * len(all_negative_tweets))

EMO = re.compile(r"(?::|;|=)(?:-)?(?:\)|\(|D|P|p)+|\)+\s*$|\(+\s*$|:'\(|:\w\b|\(:|\):|<3")
clean = [re.sub(r"\s+", " ", EMO.sub(" ", t)).strip() for t in tweets]
print('tweets that contained an emoticon:', sum(a != b for a, b in zip(tweets, clean)), 'of', len(tweets))
print(repr(tweets[0][-35:]), '->', repr(clean[0][-35:]))

# One-hot (multi-hot) encoding

The vector of a *tweet* below has a 1 for every word that occurs in it. Strictly this is a binary bag of words (a *multi-hot* vector); a one-hot vector belongs to a single word. The vocabulary is built from the **training** tweets only and in a fixed order.

In [ ]:
def build_vocab(token_lists):
    return {w: i for i, w in enumerate(sorted({w for t in token_lists for w in t}))}

def multi_hot(tokens, vocab):
    v = np.zeros(len(vocab), dtype=np.int8)
    for w in tokens:
        if w in vocab:
            v[vocab[w]] = 1
    return v

def run_multi_hot(texts):
    toks = [word_tokenize(t.lower()) for t in texts]
    tr, te = train_test_split(np.arange(len(texts)), test_size=0.2, random_state=SEED, stratify=y)
    vocab = build_vocab([toks[i] for i in tr])
    Xtr = np.array([multi_hot(toks[i], vocab) for i in tr])
    Xte = np.array([multi_hot(toks[i], vocab) for i in te])
    clf = LogisticRegression(max_iter=1000).fit(Xtr, y[tr])
    return accuracy_score(y[te], clf.predict(Xte)), clf, vocab

acc_leak, clf_leak, vocab_leak = run_multi_hot(tweets)
acc_clean, _, _ = run_multi_hot(clean)
print('emoticons kept   :', round(acc_leak, 4))
print('emoticons removed:', round(acc_clean, 4))
inv = {j: w for w, j in vocab_leak.items()}
w = clf_leak.coef_[0]
print('strongest weights when emoticons are kept:',
      [(inv[j], round(float(w[j]), 1)) for j in np.argsort(-np.abs(w))[:4]])

# Bag of words and TF-IDF

scikit-learn's default tokenizer (`\\b\\w\\w+\\b`) drops punctuation, so these two pipelines never see the emoticons. We use the cleaned text anyway and keep the same split.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def run_vec(vec, texts=clean, model=None):
    tr, te = train_test_split(np.arange(len(texts)), test_size=0.2, random_state=SEED, stratify=y)
    Xtr = vec.fit_transform([texts[i] for i in tr])      # fit on the training part only
    Xte = vec.transform([texts[i] for i in te])
    m = (LogisticRegression(max_iter=1000) if model is None else model).fit(Xtr, y[tr])
    pred = m.predict(Xte)
    return accuracy_score(y[te], pred), y[te], pred, len(vec.vocabulary_)

for name, vec in [('BoW, stop words removed', CountVectorizer(stop_words='english')),
                  ('TF-IDF, stop words removed', TfidfVectorizer(stop_words='english')),
                  ('TF-IDF, stop words kept', TfidfVectorizer()),
                  ('TF-IDF, unigrams + bigrams', TfidfVectorizer(ngram_range=(1, 2), min_df=2))]:
    acc, _, _, V = run_vec(vec)
    print(f'{name:30s} vocab={V:6d}  accuracy={acc:.4f}')
acc, yt, pred, _ = run_vec(TfidfVectorizer(ngram_range=(1, 2), min_df=2))
print(classification_report(yt, pred))

# Other classifiers on TF-IDF

`SVC(kernel='linear', probability=True)` of the original trains five extra models for the probabilities and is slow; `LinearSVC` solves the same linear problem quickly. k-NN uses the cosine distance, which suits TF-IDF vectors.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic regression': LogisticRegression(max_iter=1000),
    'Linear SVM': LinearSVC(),
    'Naive Bayes': MultinomialNB(),
    'k-NN (k=5, cosine)': KNeighborsClassifier(5, metric='cosine', algorithm='brute'),
    'Decision tree': DecisionTreeClassifier(random_state=SEED),
    'Random forest': RandomForestClassifier(100, random_state=SEED, n_jobs=-1),
}
for name, m in models.items():
    acc = run_vec(TfidfVectorizer(stop_words='english'), model=m)[0]
    print(f'{name:22s} {acc:.4f}')

# Word embeddings (GloVe)

The original cell called `!wget ... glove.6B.zip` and crashed (`Input/output error`) on a read-only file system; the file is also 822 MB. `gensim` downloads the 50-dimensional vectors (66 MB) and caches them. A tweet is represented by the **mean** of its word vectors, so the word order is lost again.

In [ ]:
try:
    import gensim.downloader as api
    kv = api.load('glove-wiki-gigaword-50')
except Exception as e:                       # no gensim or no network
    kv = None
    print('GloVe not available:', e)

if kv is not None:
    def tweet_vector(t):
        v = [kv[w] for w in word_tokenize(t.lower()) if w in kv]
        return np.mean(v, axis=0) if v else np.zeros(kv.vector_size)
    E = np.array([tweet_vector(t) for t in clean])
    tr, te = train_test_split(np.arange(len(E)), test_size=0.2, random_state=SEED, stratify=y)
    clf = LogisticRegression(max_iter=2000).fit(E[tr], y[tr])
    print('GloVe-50 mean vector + LR:', round(accuracy_score(y[te], clf.predict(E[te])), 4))
    print('most similar to king - man + woman:',
          [w for w, _ in kv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)])

# Contextual embeddings (BERT)

The model is stored in a variable called `bert` (the original overwrote `model`, the classifier). `bert.eval()` switches off dropout. The cell needs the `transformers` package and a download of about 440 MB, so it is wrapped in `try`.

In [ ]:
try:
    import torch
    from transformers import BertTokenizer, BertModel
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    bert = BertModel.from_pretrained('bert-base-uncased').eval()

    def token_vectors(sentence):
        enc = tokenizer(sentence, return_tensors='pt')
        with torch.no_grad():
            h = bert(**enc).last_hidden_state[0]            # (tokens, 768)
        return tokenizer.convert_ids_to_tokens(enc['input_ids'][0]), h

    toks, h = token_vectors('I am learning Natural Language Processing now')
    print('dimension of each vector:', h.shape[1])
    for t, v in zip(toks, h):
        print(f'{t:12s}', [round(float(x), 3) for x in v[:5]])

    # the same word in two contexts gets two different vectors
    t1, h1 = token_vectors('He sat on the river bank.')
    t2, h2 = token_vectors('She deposited the money in the bank.')
    cos = torch.nn.functional.cosine_similarity(h1[t1.index('bank')], h2[t2.index('bank')], dim=0)
    print('cosine between the two "bank" vectors:', round(float(cos), 3))
except Exception as e:
    print('BERT cell skipped:', type(e).__name__, e)